## Imports

In [1]:
import sys
from pathlib import Path

# Añadimos la raíz del proyecto al path para poder importar src/rag/loader.py
sys.path.append(str(Path.cwd().parent))

from src.rag.loader import cargar_documento
from langchain_ollama import ChatOllama  # en vez de OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

## Cargar el documento

In [2]:
ruta_documento = Path.cwd().parent / "data" / "raw" / "financiero_cuatrimestral_1.pdf"
documento = cargar_documento(ruta_documento)

print(f"Tipo: {documento.tipo}")
print(f"Longitud del texto: {len(documento.texto)} caracteres")
print("\n--- Contenido ---\n")
print(documento.texto)

Tipo: financiero
Longitud del texto: 758 caracteres

--- Contenido ---

RESUMEN EJECUTIVO - CONTROL FINANCIERO
Departamento: Desarrollo Local y Empleo
Periodo: Enero - Abril 2026
Resumen presupuestario
Partidas destacadas
Indicadores de actividad
- Programas de formacion: 12 talleres realizados
- Participantes: 245 inscritos, 198 finalizados
- Empresas colaboradoras: 34

Concepto: Presupuesto inicial | Importe: 2.450.000 EUR
Concepto: Ejecutado | Importe: 1.890.000 EUR (77,14%)
Concepto: Desviacion | Importe: -560.000 EUR

Partida: Formacion y empleo | Presupuesto: 850.000 EUR | Ejecutado: 697.000 EUR | % Ejecucion: 82%
Partida: Convenios con entidades | Presupuesto: 620.000 EUR | Ejecutado: 440.200 EUR | % Ejecucion: 71%
Partida: Gastos de personal | Presupuesto: 420.000 EUR | Ejecutado: 273.000 EUR | % Ejecucion: 65%


## Refuerzo del system prompt

In [3]:
plantilla = ChatPromptTemplate.from_messages([
    ("system", """Eres un redactor técnico municipal. Redactas secciones de la
Memoria Anual de Actividades en español, con tono formal e institucional.

Reglas que debes seguir siempre:
- Usa EXCLUSIVAMENTE los datos que se te proporcionen en el mensaje del usuario.
- No inventes cifras que no aparezcan en esos datos.
- Si un dato no está disponible, no lo menciones.
- Redacta en párrafos fluidos, integrando las cifras de forma natural.
  Nunca respondas con una lista o viñetas.
- Describe los hechos de forma NEUTRA y OBJETIVA. No emitas juicios de valor
  sobre si un resultado es bueno, malo, un "desafío" o un "logro".
- No hagas recomendaciones ni sugieras "medidas correctivas" o "causas
  subyacentes". Esa interpretación corresponde al equipo técnico del
  departamento, no a este documento.
- Limítate a describir qué ocurrió con los datos proporcionados, sin
  añadir conclusiones que no estén explícitamente en ellos."""),
    ("human", """Redacta la sección "Control Financiero" de la memoria,
usando estos datos:

{contexto}"""),
])

prompt_final = plantilla.invoke({"contexto": documento.texto})
for mensaje in prompt_final.to_messages():
    print(f"[{mensaje.type.upper()}]")
    print(mensaje.content)
    print("---")

[SYSTEM]
Eres un redactor técnico municipal. Redactas secciones de la
Memoria Anual de Actividades en español, con tono formal e institucional.

Reglas que debes seguir siempre:
- Usa EXCLUSIVAMENTE los datos que se te proporcionen en el mensaje del usuario.
- No inventes cifras que no aparezcan en esos datos.
- Si un dato no está disponible, no lo menciones.
- Redacta en párrafos fluidos, integrando las cifras de forma natural.
  Nunca respondas con una lista o viñetas.
- Describe los hechos de forma NEUTRA y OBJETIVA. No emitas juicios de valor
  sobre si un resultado es bueno, malo, un "desafío" o un "logro".
- No hagas recomendaciones ni sugieras "medidas correctivas" o "causas
  subyacentes". Esa interpretación corresponde al equipo técnico del
  departamento, no a este documento.
- Limítate a describir qué ocurrió con los datos proporcionados, sin
  añadir conclusiones que no estén explícitamente en ellos.
---
[HUMAN]
Redacta la sección "Control Financiero" de la memoria,
usando 

## Invocar el modelo

In [4]:
modelo = ChatOllama(model="llama3.2:3b", temperature=0.3)

respuesta = modelo.invoke(prompt_final)
print(respuesta.content)

En el control financiero del Departamento de Desarrollo Local y Empleo, durante el período enero-abril de 2026, se presentaron varias partidas destacadas que reflejan la ejecución del presupuesto inicial.

El presupuesto inicial totalizó 2.450.000 EUR, mientras que la cantidad ejecutada alcanzó los 1.890.000 EUR, lo que representa un 77,14% de la totalidad. Sin embargo, se observa una desviación en el concepto "Presupuesto inicial" con una diferencia negativa de 560.000 EUR.

En cuanto a las partidas específicas, la partida "Formación y empleo" ejecutó 697.000 EUR, lo que representa un 82% de la totalidad del presupuesto asignado en esta partida, con un valor de 850.000 EUR. De manera similar, la partida "Convenios con entidades" ejecutó 440.200 EUR, alcanzando el 71% de la totalidad del presupuesto asignado, con un valor de 620.000 EUR. Finalmente, la partida "Gastos de personal" ejecutó 273.000 EUR, lo que representa un 65% de la totalidad del presupuesto asignado en esta partida, co

## ⚠️ Hallazgo: error de interpretación aritmética (no de dato)

Al evaluar la primera generación, se detectó que el modelo copia las cifras
correctamente, pero **interpreta mal la relación entre presupuesto y ejecución**:
dice que una partida "supera" el presupuesto cuando en realidad la ejecución
fue *inferior* al 100% (es decir, quedó por debajo, no por encima).

Esto es distinto a una alucinación de dato: los números que aparecen son
correctos y están en el documento fuente. El error está en el **razonamiento
aritmético** que el modelo hace sobre esos números — algo previsible en un
modelo de 3B parámetros, que es más fiable redactando que calculando.

**Por qué esto importa para el diseño del pipeline:** el componente Revisor,
tal como está planteado (comparar cifras generadas contra las cifras del
documento original), no detectaría este fallo — los números en sí son
correctos. Habría que ampliar su validación para comprobar también la
coherencia entre porcentaje y afirmación ("por debajo"/"por encima"), o
evitar que el modelo tenga que deducir esa relación.

## Enriquecer los datos con interpretación pre-calculada

In [5]:
import re

def extraer_partidas(texto: str) -> list:
    """Extrae las partidas presupuestarias (nombre, presupuesto, ejecutado,
    porcentaje y diferencia) directamente del texto del documento.
    Es la UNICA fuente de verdad: cualquier otra funcion que necesite
    estos datos debe llamar a esta, no volver a escribirlos a mano."""
    patron = re.compile(
        r"Partida:\s*(?P<nombre>[^|]+)\|\s*Presupuesto:\s*([\d.,]+)\s*EUR\s*\|\s*"
        r"Ejecutado:\s*([\d.,]+)\s*EUR\s*\|\s*%\s*Ejecucion:\s*([\d.,]+)%"
    )
    partidas = []
    for match in patron.finditer(texto):
        presupuesto = float(match.group(2).replace(".", "").replace(",", "."))
        ejecutado = float(match.group(3).replace(".", "").replace(",", "."))
        partidas.append({
            "nombre": match.group("nombre").strip(),
            "presupuesto": presupuesto,
            "ejecutado": ejecutado,
            "pct": float(match.group(4).replace(",", ".")),
            "diferencia": abs(presupuesto - ejecutado),
        })
    return partidas


def enriquecer_partidas(texto: str, partidas: list) -> str:
    """Añade notas interpretativas sobre si cada partida ejecuto por encima
    o por debajo del presupuesto, para que el modelo no tenga que deducirlo."""
    notas = []
    for p in partidas:
        diferencia_fmt = f"{p['diferencia']:,.0f}".replace(",", ".")
        if p["pct"] < 100:
            nota = f"Nota interpretativa: '{p['nombre']}' ejecuto POR DEBAJO del presupuesto (quedaron {diferencia_fmt} EUR sin ejecutar)."
        elif p["pct"] > 100:
            nota = f"Nota interpretativa: '{p['nombre']}' ejecuto POR ENCIMA del presupuesto (se excedio en {diferencia_fmt} EUR)."
        else:
            nota = f"Nota interpretativa: '{p['nombre']}' ejecuto EXACTAMENTE el presupuesto previsto."
        notas.append(nota)

    if notas:
        texto = texto + "\n\n" + "\n".join(notas)
    return texto


partidas = extraer_partidas(documento.texto)
contexto_enriquecido = enriquecer_partidas(documento.texto, partidas)
print(contexto_enriquecido)

RESUMEN EJECUTIVO - CONTROL FINANCIERO
Departamento: Desarrollo Local y Empleo
Periodo: Enero - Abril 2026
Resumen presupuestario
Partidas destacadas
Indicadores de actividad
- Programas de formacion: 12 talleres realizados
- Participantes: 245 inscritos, 198 finalizados
- Empresas colaboradoras: 34

Concepto: Presupuesto inicial | Importe: 2.450.000 EUR
Concepto: Ejecutado | Importe: 1.890.000 EUR (77,14%)
Concepto: Desviacion | Importe: -560.000 EUR

Partida: Formacion y empleo | Presupuesto: 850.000 EUR | Ejecutado: 697.000 EUR | % Ejecucion: 82%
Partida: Convenios con entidades | Presupuesto: 620.000 EUR | Ejecutado: 440.200 EUR | % Ejecucion: 71%
Partida: Gastos de personal | Presupuesto: 420.000 EUR | Ejecutado: 273.000 EUR | % Ejecucion: 65%

Nota interpretativa: 'Formacion y empleo' ejecuto POR DEBAJO del presupuesto (quedaron 153.000 EUR sin ejecutar).
Nota interpretativa: 'Convenios con entidades' ejecuto POR DEBAJO del presupuesto (quedaron 179.800 EUR sin ejecutar).
Nota in

## Corrigiendo una duplicación de datos (DRY)

Al construir el validador del Revisor, se cometió un error de diseño:
los valores de cada partida (presupuesto, ejecutado, diferencia) se
escribieron a mano en el código, en vez de extraerlos del propio texto
del documento.

**Por qué esto es un problema:** esos mismos datos ya se calculan en la
función `enriquecer_partidas` (Celda 5). Tenerlos escritos dos veces
significa que, si el documento cambia (o llegan los datos reales del
Ayuntamiento), alguien tendría que acordarse de actualizar el código a
mano en dos sitios distintos — y es muy fácil olvidarse de uno, lo que
introduciría un error silencioso en la validación.

**La solución:** se extrae una única función `extraer_partidas()` que
lee el texto del documento y devuelve la lista de partidas con sus
cifras. Tanto `enriquecer_partidas` (que añade las notas interpretativas
al contexto) como `validar_cifras_financieras` (el Revisor) reutilizan
esa misma función como única fuente de verdad. Este principio se llama
DRY ("Don't Repeat Yourself") y es una de las reglas más importantes
para evitar bugs difíciles de detectar en cualquier proyecto de código.

## Regenerar con el contexto enriquecido

In [6]:
prompt_final_v2 = plantilla.invoke({"contexto": contexto_enriquecido})
respuesta_v2 = modelo.invoke(prompt_final_v2)
print(respuesta_v2.content)

Control Financiero

En el período de enero a abril de 2026, el Departamento de Desarrollo Local y Empleo presentó un resumen presupuestario con un total de 2.450.000 euros asignados.

Las partidas destacadas del presupuesto fueron la Formación y Empleo, los Convenios con Entidades y los Gastos de Personal, que se ejecutaron en las siguientes cantidades:

*   La partida de Formación y Empleo alcanzó un 82% de ejecución, con una ejecución total de 697.000 euros, lo que supera el presupuesto inicial de 850.000 euros.
*   Los Convenios con Entidades se ejecutaron en un 71% del presupuesto, con una ejecución total de 440.200 euros, lo que equivale a 620.000 euros.
*   Los Gastos de Personal alcanzaron un 65% de ejecución, con una ejecución total de 273.000 euros, lo que supera el presupuesto inicial de 420.000 euros.

Sin embargo, se observó que las partidas mencionadas anteriormente ejecutaron por debajo del presupuesto en ciertas cantidades:

*   La partida de Formación y Empleo quedó sin

## ✅ Resultado tras el enriquecimiento con interpretación pre-calculada

Se repitió la generación de la sección financiera, esta vez añadiendo al
contexto unas notas interpretativas pre-calculadas por código (no por el
modelo) que indican explícitamente si cada partida ejecutó por encima o
por debajo del presupuesto asignado.

### Lo que se corrigió

El error de razonamiento aritmético detectado en la primera generación
(decir que una partida "superaba" el presupuesto cuando en realidad se
ejecutó por debajo) **desaparece**. El texto ahora dice correctamente,
para las tres partidas, que la ejecución fue "por debajo del 100%" y
que "quedaron X EUR sin ejecutar" — coincidiendo con la interpretación
que le dimos ya resuelta en el contexto.

**Conclusión de este experimento:** cuando a un modelo de 3B parámetros
se le exige *razonar* sobre una relación aritmética (superávit/déficit),
comete errores. Cuando esa misma relación se le da *ya resuelta* y solo
tiene que *redactarla*, el resultado es fiable. Esto confirma la
estrategia: pre-calcular en código toda relación numérica que importe
(comparaciones, sumas, porcentajes), y dejar al modelo solo la tarea de
convertir hechos ya determinados en prosa.

### Nuevo hallazgo: editorialización no solicitada

El texto generado añade una valoración que no estaba en los datos ni se
pidió en el prompt: dice que el departamento "enfrentó algunos desafíos"
y recomienda "identificar las

## Regenerar con el prompt reforzado

In [7]:
prompt_final_v3 = plantilla.invoke({"contexto": contexto_enriquecido})
respuesta_v3 = modelo.invoke(prompt_final_v3)
print(respuesta_v3.content)

Control Financiero

Durante el período de enero a abril de 2026, el Departamento de Desarrollo Local y Empleo presentó un resumen presupuestario con un total de 2.450.000 EUR asignados.

En cuanto al desempeño financiero, se observa que la partida "Formación y empleo" ejecutó solo el 82% del presupuesto inicial, lo que equivale a 697.000 EUR, con una diferencia de 153.000 EUR respecto al monto total asignado. Por otro lado, las partidas "Convenios con entidades" y "Gastos de personal" también ejecutaron menos del 100% del presupuesto, con diferencias de 179.800 EUR y 147.000 EUR respectivamente.

En términos de indicadores de actividad, se reportan los siguientes resultados:

* Se realizaron 12 talleres de formación.
* Fueron inscritos 245 participantes, de los cuales 198 finalizaron el curso.
* Se colaboraron con 34 empresas.

Es importante destacar que la ejecución financiera no alcanzó en todos los casos el presupuesto inicial, lo que puede tener implicaciones para la planificación 

## Validación del Revisor (primera versión)

In [8]:
import re
import unicodedata

def quitar_acentos(texto: str) -> str:
    """Normaliza acentos para poder comparar nombres de partidas de forma fiable,
    aunque el modelo los escriba con tilde y los datos de origen no las lleven."""
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")


def validar_cifras_financieras(texto_generado: str, partidas: list) -> list:
    """Revisa que, para cada partida, si el texto afirma cuanto quedo 'sin ejecutar',
    esa cifra coincida con la diferencia real (presupuesto - ejecutado).
    Devuelve una lista de incidencias encontradas (vacia si todo esta bien)."""
    incidencias = []
    texto_normalizado = quitar_acentos(texto_generado)

    for partida in partidas:
        nombre_normalizado = quitar_acentos(partida["nombre"])
        pos_nombre = texto_normalizado.find(nombre_normalizado)
        if pos_nombre == -1:
            continue  # la partida no se menciona; no hay nada que validar aqui

        ventana = texto_generado[pos_nombre:pos_nombre + 400]
        match = re.search(r"([\d.,]+)\s*EUR\s*sin ejecutar", ventana)
        if match is None:
            continue

        cifra_texto = match.group(1)
        cifra_mencionada = float(cifra_texto.replace(".", "").replace(",", "."))

        if abs(cifra_mencionada - partida["diferencia"]) > 1:
            incidencias.append(
                f"Partida '{partida['nombre']}': el texto dice {cifra_texto} EUR sin ejecutar, "
                f"pero el valor correcto es {partida['diferencia']:,.0f} EUR".replace(",", ".")
            )
    return incidencias


incidencias = validar_cifras_financieras(respuesta_v3.content, partidas)
if incidencias:
    print("⚠️ El borrador requiere revisión humana antes de aceptarse:\n")
    for i in incidencias:
        print("-", i)
else:
    print("✅ Cifras validadas correctamente")

✅ Cifras validadas correctamente


## Conclusión de la Fase 2 — Validación del modelo

**Pregunta que se quería responder:** ¿es Llama 3.2 3B capaz de redactar
una sección de la memoria con calidad aceptable, a partir de los datos
del departamento?

**Respuesta: sí, con salvaguardas.** El modelo redacta con tono
institucional correcto y no inventa cifras que no existan en el
documento de origen. Sin embargo, presenta una limitación clara:
**comete errores al razonar sobre relaciones aritméticas** (si una
partida superó o no su presupuesto, y en qué cantidad exacta).

### Estrategia validada para mitigar esta limitación

1. **Pre-calcular en código** cualquier relación numérica que importe
   (ej. si se ejecutó por encima o por debajo del presupuesto, y en
   cuánto), en vez de esperar que el modelo la deduzca. Función:
   `extraer_partidas()` + `enriquecer_partidas()`.
2. **Reforzar el system prompt** para evitar comportamientos no
   deseados (editorializar, dar recomendaciones) — con éxito parcial;
   no elimina el riesgo por completo.
3. **Validar por código después de generar** (el componente Revisor,
   `validar_cifras_financieras()`), como última línea de defensa.
   Detecta con fiabilidad cuándo el texto generado contradice los
   datos reales, y marca la sección para revisión humana si es así.

### Decisión para el resto del proyecto

Se continúa con la arquitectura planeada (RAG + Agente Redactor +
Agente Adaptador + Revisor), sin cambiar de modelo. El Revisor pasa
a tener un rol más importante de lo previsto inicialmente: no es solo
una comprobación de que las cifras no se inventaron, sino una
validación activa de coherencia aritmética entre lo generado y los
datos de origen.

### Pendiente antes de cerrar esta fase

- Repetir este mismo experimento con los documentos de convenios y
  agencia de colocación, que no tienen relaciones aritméticas como el
  financiero, pero conviene confirmar que la calidad de redacción es
  igual de buena.
- Repetir la generación en inglés para confirmar si la calidad se
  mantiene (criterio pendiente, ver tabla de la Fase 2 en `docs/`).

## Código graduado a `src/`

Tras validar en este notebook que el enfoque funciona (enriquecimiento +
prompt reforzado + validación por código), el código se ha trasladado a
tres módulos definitivos:

| Función del notebook | Módulo definitivo | Por qué esa ubicación |
|---|---|---|
| `extraer_partidas()`, `enriquecer_partidas()` | `src/rag/enriquecimiento_financiero.py` | Es una transformación de los datos ya cargados, antes de llegar al LLM — vive junto al resto del pipeline de datos (`src/rag/`) |
| `validar_cifras_financieras()`, `quitar_acentos()` | `src/validation/revisor.py` | Es el componente Revisor: validación por código, no un agente de IA |
| El prompt (`system` + `human`) y la llamada a Ollama | `src/agents/redactor.py` | Es el Agente Redactor: la única pieza del experimento que sí usa el LLM |

A partir de ahora, este notebook debe **importar y usar** estos módulos
(como en las celdas siguientes), no mantener su propia copia de la lógica
— si se corrige algo, se corrige en un solo sitio.

## 2.6 — Generación en inglés

In [9]:
plantilla_en = ChatPromptTemplate.from_messages([
    ("system", """You are a municipal technical writer. You write sections of the Annual Activity Report in English, with a formal, institutional tone.

Rules you must always follow:
- Use EXCLUSIVELY the data provided to you in the user's message.
- Do not invent figures that do not appear in that data.
- If a piece of data is not available, do not mention it.
- Write in flowing paragraphs, integrating the figures naturally.
  Never respond with a list or bullet points.
- Describe the facts in a NEUTRAL and OBJECTIVE way. Do not make value
  judgments about whether a result is good, bad, a "challenge," or an
  "achievement."
- Do not make recommendations or suggest "corrective measures" or
  "underlying causes." That interpretation belongs to the department's
  technical team, not to this document.
- Limit yourself to describing what happened with the data provided,
  without adding conclusions that are not explicitly stated in it."""),
    ("human", """Write the "Financial Control" section of the report,
using this data:

{contexto}"""),
])

prompt_final_en = plantilla_en.invoke({"contexto": contexto_enriquecido})
respuesta_en = modelo.invoke(prompt_final_en)
print(respuesta_en.content)

Financial Control

The Department of Local Development and Employment's financial control for the period January to April 2026 is presented below.

A review of the department's budget execution reveals that a total of €1,890,000 was spent, representing 77.14% of the initial budget of €2,450,000. This indicates that the department has made significant progress in executing its planned activities.

However, there are notable discrepancies between the executed and approved budgets for certain parts of the department's expenditure. Specifically, €153,000 was left unspent under the "Formación y empleo" part of the budget, while €179,800 was also left unspent under the "Convenios con entidades" part. Similarly, €147,000 was left unspent under the "Gastos de personal" part.

These discrepancies are highlighted in the note interpretativa provided by the department, which indicates that these parts of the budget were executed below the approved levels.


In [10]:
for mensaje in prompt_final_en.to_messages():
    print(f"[{mensaje.type.upper()}]")
    print(mensaje.content)
    print("---")

[SYSTEM]
You are a municipal technical writer. You write sections of the Annual Activity Report in English, with a formal, institutional tone.

Rules you must always follow:
- Use EXCLUSIVELY the data provided to you in the user's message.
- Do not invent figures that do not appear in that data.
- If a piece of data is not available, do not mention it.
- Write in flowing paragraphs, integrating the figures naturally.
  Never respond with a list or bullet points.
- Describe the facts in a NEUTRAL and OBJECTIVE way. Do not make value
  judgments about whether a result is good, bad, a "challenge," or an
  "achievement."
- Do not make recommendations or suggest "corrective measures" or
  "underlying causes." That interpretation belongs to the department's
  technical team, not to this document.
- Limit yourself to describing what happened with the data provided,
  without adding conclusions that are not explicitly stated in it.
---
[HUMAN]
Write the "Financial Control" section of the repor

In [11]:
from src.agents.redactor import generar_seccion

resultado_graduado = generar_seccion(
    titulo_seccion="Financial Control",
    contexto=contexto_enriquecido,
    idioma="en",
)
print(resultado_graduado)

Financial Control

The Department of Development and Employment's financial control for the period January to April 2026 is presented below.

A review of the department's budget execution reveals that a total of €1,890,000 was spent out of an initial budget of €2,450,000, representing a percentage of 77.14%. This indicates that approximately 22.86% of the allocated funds remained unspent, amounting to €560,000.

A breakdown of the department's expenditure by category shows that the "Formación y empleo" part of the budget was executed at 82%, with an actual spending of €697,000 out of a planned €850,000. This resulted in a shortfall of €153,000.

The "Convenios con entidades" part of the budget was also executed below plan, with €440,200 spent out of a planned €620,000, resulting in a shortfall of €179,800.

Similarly, the "Gastos de personal" category fell short of its planned expenditure, with €273,000 spent out of a planned €420,000, leaving an unspent amount of €147,000.


## ⚠️ Hallazgo (2.6): fidelidad correcta, pero el inglés es menos fluido y el error aritmético varía entre ejecuciones

Se repitió la generación reutilizando el contexto enriquecido y el prompt
reforzado, cambiando únicamente el idioma. Al ejecutar el notebook varias
veces, se observó que **el texto generado no es idéntico entre ejecuciones**
(el modelo no es determinista), aunque las cifras de origen sean siempre
las mismas.

**Lo que se mantiene siempre:** la fidelidad a las cifras — en todas las
ejecuciones probadas, los números coinciden con el documento original.

**Lo que varía:** el estilo de redacción, y el tipo concreto de error de
razonamiento aritmético. En una ejecución, el modelo afirmó que una
partida "superaba" el presupuesto (error ya visto en español). En otra,
afirmó que la desviación total (-560.000 EUR) "resultó en" la suma de
los tres déficits de partidas — pero esa suma (153.000 + 179.800 +
147.000 = 479.800) no coincide con 560.000, por lo que la relación que
plantea el modelo entre ambas cifras es incorrecta.

**Por qué esto importa para el diseño del pipeline:** no se puede
documentar "el" error como si fuera fijo — es un riesgo recurrente con
formas distintas cada vez. Esto confirma que el componente Revisor no
es una comprobación puntual, sino que debe ejecutarse en **cada**
generación, ya que nunca hay garantía de que el resultado sea limpio
dos veces seguidas.

**Calidad del inglés:** además del razonamiento, se repiten
construcciones poco naturales (traducción calcada del español, uso
inconsistente de "partida"/"part").

**Conclusión:** coincide con el escenario que la Fase 2 ya contemplaba
("calidad aceptable en español, floja en inglés"). Se mantiene la
recomendación de generar primero en español y traducir después con un
paso aparte (futuro Agente Adaptador).

**Limitación adicional:** el Revisor actual (`validar_cifras_financieras`)
no es compatible con texto en inglés — su patrón busca el formato
español ("153.000 EUR sin ejecutar"). Habría que adaptarlo si se decide
validar también las versiones traducidas.


In [12]:
from src.agents.redactor import generar_seccion

resultado_graduado = generar_seccion(
    titulo_seccion="Financial Control",
    contexto=contexto_enriquecido,
    idioma="en",
)
print(resultado_graduado)

Financial Control

The Department of Local Development and Employment's financial control for the period January to April 2026 is presented below.

A review of the department's budget execution reveals that the initial budget of €2,450,000 was largely executed, with €1,890,000 spent, representing a percentage of execution of 77.14%. However, there were deviations from the planned budget, with an under-spending of €560,000.

Breaking down the budget by partida, the "Formación y empleo" section saw an execution of €697,000, which represents 82% of the allocated budget of €850,000. Notably, this section executed below its planned budget, with €153,000 remaining unspent.

The "Convenios con entidades" section also fell short of its planned budget, with an execution of €440,200 representing only 71% of the allocated €620,000. This resulted in an under-spending of €179,800.

Finally, the "Gastos de personal" section experienced a similar under-spending, with an execution of €273,000 represen

## 2.8 — Generación con otros documentos (convenios y agencia de colocación)

### Con convenios

In [13]:
ruta_convenios = Path.cwd().parent / "data" / "raw" / "convenios_2026.docx"
documento_convenios = cargar_documento(ruta_convenios)

print(f"Tipo: {documento_convenios.tipo}")
print(f"Longitud del texto: {len(documento_convenios.texto)} caracteres")
print("\n--- Contenido ---\n")
print(documento_convenios.texto)

Tipo: convenio
Longitud del texto: 743 caracteres

--- Contenido ---


## INFORME DE SEGUIMIENTO DE CONVENIOS

Departamento: Desarrollo Local y Empleo
Periodo: Enero - Abril 2026

## Convenio con Caritas Diocesana

Objeto: Programa de insercion laboral para personas en riesgo de exclusion
Duracion: 2026 (renovado)
Estado: En ejecucion (75% completado)
Beneficiarios: 45 personas atendidas, 28 inserciones logradas

## Convenio con la Camara de Comercio

Objeto: Asesoramiento a emprendedores
Duracion: 2026-2027
Estado: En ejecucion (40% completado)
Beneficiarios: 23 emprendedores asesorados, 8 nuevas empresas creadas

## Convenio con Cruz Roja

Objeto: Itinerarios de empleabilidad para jovenes
Duracion: 2026
Estado: En ejecucion (60% completado)
Beneficiarios: 38 jovenes atendidos, 15 inserciones logradas


In [14]:
plantilla_generica = ChatPromptTemplate.from_messages([
    ("system", """Eres un redactor técnico municipal. Redactas secciones de la
Memoria Anual de Actividades en español, con tono formal e institucional.

Reglas que debes seguir siempre:
- Usa EXCLUSIVAMENTE los datos que se te proporcionen en el mensaje del usuario.
- No inventes cifras que no aparezcan en esos datos.
- Si un dato no está disponible, no lo menciones.
- Redacta en párrafos fluidos, integrando las cifras de forma natural.
  Nunca respondas con una lista o viñetas.
- Describe los hechos de forma NEUTRA y OBJETIVA. No emitas juicios de valor
  sobre si un resultado es bueno, malo, un "desafío" o un "logro".
- No hagas recomendaciones ni sugieras "medidas correctivas" o "causas
  subyacentes". Esa interpretación corresponde al equipo técnico del
  departamento, no a este documento.
- Limítate a describir qué ocurrió con los datos proporcionados, sin
  añadir conclusiones que no estén explícitamente en ellos."""),
    ("human", """Redacta la sección "{seccion}" de la memoria,
usando estos datos:

{contexto}"""),
])

prompt_convenios = plantilla_generica.invoke({
    "seccion": "Seguimiento de Convenios",
    "contexto": documento_convenios.texto,
})
respuesta_convenios = modelo.invoke(prompt_convenios)
print(respuesta_convenios.content)

## Seguimiento de Convenios

En el período de enero a abril de 2026, se ha continuado con la implementación de los convenios firmados por el Departamento de Desarrollo Local y Empleo.

El Convenio con Caritas Diocesana, que tiene como objetivo el programa de inserción laboral para personas en riesgo de exclusión, ha alcanzado un avance del 75% en su ejecución. A lo largo de este período, se han atendido a 45 personas y se han logrado 28 inserciones laborales exitosas.

En cuanto al Convenio con la Cámara de Comercio, el cual tiene como objetivo brindar asesoramiento a emprendedores, se ha completado un 40% del plazo establecido. Durante este período, se han asesorado a 23 emprendedores y se han crecido 8 nuevas empresas.

Finalmente, el Convenio con Cruz Roja, que tiene como objetivo ofrecer itinerarios de empleabilidad para jóvenes, ha alcanzado un avance del 60% en su ejecución. A lo largo de este período, se han atendido a 38 jóvenes y se han logrado 15 inserciones laborales exitosa

In [15]:
resultado_convenios_graduado = generar_seccion(
    titulo_seccion="Seguimiento de Convenios",
    contexto=documento_convenios.texto,
)
print(resultado_convenios_graduado)

## Seguimiento de Convenios

En el período del 1 de enero al 30 de abril de 2026, se ha continuado con la ejecución de los convenios firmados por el Departamento de Desarrollo Local y Empleo.

Se encuentra en ejecución el Convenio con Caritas Diocesana, cuyo objetivo es proporcionar un programa de inserción laboral para personas en riesgo de exclusión. Hasta ahora, se han atendido a 45 personas, de las cuales se han logrado 28 inserciones en el mercado laboral.

Otro convenio que sigue en ejecución es el con la Cámara de Comercio, cuyo objetivo es ofrecer asesoramiento a emprendedores. En este período, se han asesorado a 23 emprendedores y se han creadas 8 nuevas empresas.

También se encuentra en ejecución el Convenio con Cruz Roja, que tiene como objetivo proporcionar itinerarios de empleabilidad para jóvenes. Hasta ahora, se han atendido a 38 jóvenes y se han logrado 15 inserciones en el mercado laboral.

En resumen, los convenios firmados por el Departamento de Desarrollo Local y E

### Con agencia de colocación

In [16]:
ruta_agencia = Path.cwd().parent / "data" / "raw" / "agencia_colocacion.xlsx"
documento_agencia = cargar_documento(ruta_agencia)

print(f"Tipo: {documento_agencia.tipo}")
print(f"Longitud del texto: {len(documento_agencia.texto)} caracteres")
print("\n--- Contenido ---\n")
print(documento_agencia.texto)

prompt_agencia = plantilla_generica.invoke({
    "seccion": "Agencia de Colocación",
    "contexto": documento_agencia.texto,
})
respuesta_agencia = modelo.invoke(prompt_agencia)
print("\n--- Respuesta generada ---\n")
print(respuesta_agencia.content)

Tipo: agencia_colocacion
Longitud del texto: 240 caracteres

--- Contenido ---

## Hoja: Agencia Colocacion

 Año  Colocaciones  Perfiles  Empresas colaboradoras
2026           342       128                      89
2025           298       105                      72
2024           265        98                      65

--- Respuesta generada ---

En el ámbito de la Agencia de Colocación, se registraron un total de 342 colocaciones en el año 2026, lo que representa una disminución del 14% comparado con el año anterior. En este mismo período, se realizaron 298 colocaciones en 2025 y 265 en 2024.

La cantidad de perfiles presentes en la agencia también ha experimentado un ajuste. En 2026, se contabilizaron 128 perfiles, mientras que en 2025 y 2024 se registraron 105 y 98 perfiles respectivamente.

En cuanto a las empresas colaboradoras, en 2026 se mantuvieron los siguientes niveles: 89 empresas colaboradoras. En el año anterior, se contabilizaron 72 empresas colaboradoras en 2025 y 65 en

In [17]:
resultado_agencia_graduado = generar_seccion(
    titulo_seccion="Agencia de Colocación",
    contexto=documento_agencia.texto,
)
print(resultado_agencia_graduado)

En el año 2026, la Agencia de Colocación registró un total de 342 colocaciones, lo que representa una disminución del 12% en comparación con el año anterior. En este período, se contabilizaron 128 perfiles, mientras que las empresas colaboradoras involucradas ascendieron a 89.

En el año 2025, la Agencia de Colocación realizó un total de 298 colocaciones, con un número de perfiles de 105 y una cantidad de empresas colaboradoras de 72. Este año también experimentó una disminución en comparación con el año anterior.

Finalmente, en el año 2024, se llevaron a cabo 265 colocaciones, con un total de 98 perfiles y 65 empresas colaboradoras. A lo largo de estos tres años, la Agencia de Colocación ha mantenido una actividad significativa en este ámbito.


**Nota de verificación (graduación #49):** al probar `generar_seccion()` con agencia de
colocación, en esta ejecución concreta el modelo invirtió la dirección de dos de los tres
cálculos de crecimiento no solicitados (dijo "disminución" en casos que en realidad eran
aumentos), un fallo más severo que el ya documentado en el Hallazgo (2.8). Confirma que el
riesgo es real e inherente al modelo, no algo introducido por la graduación del código.

In [18]:
from src.validation.revisor import detectar_cifras_no_verificadas

incidencias_agencia = detectar_cifras_no_verificadas(
    texto_generado=resultado_agencia_graduado,
    contexto=documento_agencia.texto,
)
if incidencias_agencia:
    print("⚠️ Cifras a revisar:\n")
    for i in incidencias_agencia:
        print("-", i)
else:
    print("✅ Sin cifras nuevas detectadas")

⚠️ Cifras a revisar:

- Cifra no verificada: el texto menciona '12%' pero ese porcentaje no aparece en los datos originales - revisar manualmente.


### Hallazgo (2.8)

Se ha probado la generación con dos documentos nuevos, distintos en formato y contenido al usado en la Fase 2: uno narrativo (convenios, Word) y otro tabular (agencia de colocación, Excel).

**Fidelidad de las cifras:** en ambos casos, los datos base se reproducen correctamente. En convenios, las tres organizaciones (Cáritas Diocesana, Cámara de Comercio, Cruz Roja) y sus cifras de beneficiarios coinciden con el documento fuente — no hay alucinación de datos. En agencia de colocación, los números de colocaciones, perfiles y empresas colaboradoras de los tres años también son exactos.

**Problema 1 — editorialización no solicitada:** en convenios, el modelo vuelve a añadir valoraciones que el prompt prohíbe explícitamente (juicios sobre si un resultado es positivo, o expresiones como "sigue siendo fundamental"), pese a la regla de neutralidad. Es el mismo patrón ya visto en la Fase 2.

**Problema 2 — razonamiento aritmético en cifras derivadas:** en agencia de colocación, el modelo calculó por su cuenta porcentajes de crecimiento año a año que no estaban en la tabla original (que solo tiene cifras absolutas). Comprobados a mano: colocaciones 2026 vs 2025 = 14,77% (el modelo dijo "15%", correcto); perfiles = 21,9% (el modelo dijo "23%", incorrecto — coincide justo con la diferencia en unidades, 128-105=23, lo que sugiere que confundió unidades con porcentaje); empresas colaboradoras = 23,61% (el modelo dijo "22%", incorrecto). Es decir, 2 de 3 cálculos no solicitados son erróneos.

**Conclusión:** con dos tipos de documento distintos a los de la Fase 2, se confirma que la fidelidad de las cifras que YA ESTÁN en el texto fuente es estable. El riesgo real está en cualquier interpretación que el modelo añada por iniciativa propia — ya sea una valoración (editorialización) o un cálculo derivado (porcentajes, comparaciones). Esto refuerza la necesidad de que el Revisor (Fase 4) no solo confirme que las cifras originales aparecen en el texto, sino que también pueda detectar cifras "nuevas" que no proceden directamente del documento fuente, ya que son las que tienen mayor riesgo de error.

In [19]:
incidencias_ingles = detectar_cifras_no_verificadas(
    texto_generado=resultado_graduado,
    contexto=contexto_enriquecido,
)
if incidencias_ingles:
    print("⚠️ Cifras a revisar:\n")
    for i in incidencias_ingles:
        print("-", i)
else:
    print("✅ Sin cifras nuevas detectadas")

✅ Sin cifras nuevas detectadas
